# B2S 05 - Andinalog Inventory Tracking

Conversión Bronze a Silver de movimientos de inventario por lote, producto, centro y movimiento. Las fechas son calendario y se conservan sin desplazamiento horario.

In [1]:
import os
import platform
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)

def find_root():
    candidates = []
    if os.getenv('ANDINALOG_ROOT'):
        candidates.append(Path(os.environ['ANDINALOG_ROOT']))
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / 'datos' / 'bronze').is_dir():
            return candidate
    raise FileNotFoundError('No se encontró datos/bronze')

ROOT = find_root()
EXECUTED_AT_UTC = datetime.now(timezone.utc).isoformat()
CONFIG = {
    'entidad': 'movimiento de inventario por lote, producto, centro y movimiento',
    'granularidad': 'una fila por movimiento_id, lote_id, producto_id y centro_distribucion',
    'lectura': {'encoding': 'utf-8', 'dtype': 'str', 'keep_default_na': False},
    'rutas': {
        'bronze': 'datos/bronze/andinalog_inventory_tracking.csv',
        'silver': 'datos/silver/andinalog_inventory_tracking_silver.csv',
        'quarantine': 'datos/quarantine/andinalog_inventory_tracking_quarantine.csv',
        'informe': 'informes/bronze_silver/Informe_B2S_05_Andinalog_Inventory_Tracking.md',
        'notebook': 'notebooks/bronze_silver/05_inventory_tracking/B2S_05_Inventory_Tracking.ipynb',
        'productos_silver': 'datos/silver/andinalog_productos_silver.csv',
        'wms_silver': 'datos/silver/andinalog_wms_orders_silver.csv'
    },
    'columnas_obligatorias': ['movimiento_id', 'lote_id', 'producto_id', 'centro_distribucion', 'fecha_ingreso', 'fecha_salida', 'fecha_vencimiento', 'cantidad_ingreso', 'cantidad_salida', 'cantidad_merma', 'dias_en_almacen', 'costo_unitario_bob'],
    'numericas': ['cantidad_ingreso', 'cantidad_salida', 'cantidad_merma', 'dias_en_almacen', 'costo_unitario_bob'],
    'centros_validos': ['Cochabamba', 'La Paz', 'Santa Cruz', 'Oruro', 'Tarija'],
    'fecha_corte': '2026-08-31',
    'zonas_horarias': {'fechas_calendario': 'sin_desplazamiento', 'timestamps_sin_zona': 'America/La_Paz_a_UTC'},
    'centinelas': [-999],
    'vencimiento_wms_habilitado': True,
    'regla_wms': 'Solo lote_id con exactamente una fecha valida y consistente en Silver WMS',
    'regla_duplicados': 'Duplicado exacto: conservar una fila y enviar copias con trazabilidad; duplicado conflictivo: enviar todas las filas'
}
PATHS = {key: ROOT / value for key, value in CONFIG['rutas'].items()}
for key in ['silver', 'quarantine', 'informe']:
    PATHS[key].parent.mkdir(parents=True, exist_ok=True)
print('Raíz detectada:', ROOT)
print('Configuración centralizada para: inventory tracking, B2S 05')
print('Fecha de ejecución UTC:', EXECUTED_AT_UTC)

Raíz detectada: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2
Configuración centralizada para: inventory tracking, B2S 05
Fecha de ejecución UTC: 2026-09-25T02:50:31.441284+00:00


In [2]:
bronze = pd.read_csv(PATHS['bronze'], encoding=CONFIG['lectura']['encoding'], dtype=CONFIG['lectura']['dtype'], keep_default_na=CONFIG['lectura']['keep_default_na'])
bronze.insert(0, '_fila_bronze', range(1, len(bronze) + 1))
productos_aux = pd.read_csv(PATHS['productos_silver'], dtype=str, keep_default_na=False)
wms_aux = pd.read_csv(PATHS['wms_silver'], dtype=str, keep_default_na=False)
perfil = {'filas': len(bronze), 'columnas': bronze.columns.drop('_fila_bronze').tolist(), 'nulos_o_vacios': {c: int(bronze[c].eq('').sum()) for c in CONFIG['columnas_obligatorias']}, 'duplicados_movimiento_id': int(bronze.duplicated('movimiento_id', keep=False).sum()), 'duplicados_lote_id': int(bronze.duplicated('lote_id', keep=False).sum()), 'centros_observados': sorted(bronze['centro_distribucion'].unique().tolist()), 'productos_observados': int(bronze['producto_id'].nunique()), 'wms_tiene_lote_id': 'lote_id' in wms_aux.columns}
print('Perfil Bronze y auxiliar:')
print(perfil)
print('WMS columnas auxiliares:', list(wms_aux.columns))
print(bronze.head(5).to_string(index=False))

Perfil Bronze y auxiliar:
{'filas': 6040, 'columnas': ['movimiento_id', 'lote_id', 'producto_id', 'centro_distribucion', 'fecha_ingreso', 'fecha_salida', 'fecha_vencimiento', 'cantidad_ingreso', 'cantidad_salida', 'cantidad_merma', 'dias_en_almacen', 'costo_unitario_bob'], 'nulos_o_vacios': {'movimiento_id': 0, 'lote_id': 0, 'producto_id': 0, 'centro_distribucion': 0, 'fecha_ingreso': 0, 'fecha_salida': 0, 'fecha_vencimiento': 18, 'cantidad_ingreso': 0, 'cantidad_salida': 0, 'cantidad_merma': 0, 'dias_en_almacen': 0, 'costo_unitario_bob': 0}, 'duplicados_movimiento_id': 80, 'duplicados_lote_id': 80, 'centros_observados': ['Cochabamba', 'La Paz', 'Oruro', 'Santa Cruz', 'Tarija'], 'productos_observados': 83, 'wms_tiene_lote_id': False}
WMS columnas auxiliares: ['order_id', 'cliente_id', 'producto_id', 'fecha_despacho', 'centro_distribucion', 'camion_id', 'chofer_id', 'cantidad_solicitada', 'cantidad_entregada', 'tiempo_entrega_prometido_hrs', 'tiempo_entrega_real_hrs', 'otif_on_time', 'o

In [3]:
def preparar(df):
    out = df.copy()
    for col in CONFIG['columnas_obligatorias']:
        out[col + '_original'] = out[col]
    out['errores_bloqueantes'] = ''
    out['motivos_transformacion'] = ''
    out['motivos_imputacion'] = ''
    out['fue_transformada'] = False
    out['fue_imputada'] = False
    return out

def add_error(df, mask, motivo):
    out = df.copy()
    mask = mask.fillna(False)
    current = out.loc[mask, 'errores_bloqueantes']
    out.loc[mask, 'errores_bloqueantes'] = np.where(current.eq(''), motivo, current + ';' + motivo)
    return out

def normalizar_texto(df):
    out = df.copy()
    for col in ['movimiento_id', 'lote_id', 'producto_id', 'centro_distribucion']:
        out[col] = out[col].str.strip()
    out['producto_id'] = out['producto_id'].str.upper()
    out['movimiento_id'] = out['movimiento_id'].str.upper()
    out['lote_id'] = out['lote_id'].str.upper()
    for col in ['movimiento_id', 'lote_id', 'producto_id', 'centro_distribucion']:
        out['fue_transformada'] |= out[col].ne(out[col + '_original'])
    return out

def convertir_numericas(df):
    out = df.copy()
    for col in CONFIG['numericas']:
        out[col] = pd.to_numeric(out[col].replace('', pd.NA), errors='coerce')
        invalid = out[col].isna() & out[col + '_original'].ne('')
        sentinel = out[col].isin(CONFIG['centinelas'])
        out[col + '_conversion_invalida'] = invalid
        out[col + '_centinela_detectado'] = sentinel
        out = add_error(out, invalid | sentinel, col + ':conversion_invalida_o_centinela')
        out.loc[out[col] < 0, col] = np.nan
        out = add_error(out, out[col].isna() & out[col + '_original'].ne(''), col + ':valor_no_numerico_o_negativo')
    return out

def manejar_timestamps_utc(df):
    out = df.copy()
    for col in ['fecha_ingreso', 'fecha_salida', 'fecha_vencimiento']:
        out[col + '_timestamp_detectado'] = out[col].str.contains(r'\d{1,2}:\d{2}', regex=True, na=False)
        out[col + '_timestamp_utc'] = pd.NA
        mask = out[col + '_timestamp_detectado']
        parsed = pd.to_datetime(out.loc[mask, col], format='mixed', errors='coerce')
        out.loc[mask, col + '_timestamp_utc'] = parsed.dt.tz_localize(ZoneInfo('America/La_Paz')).dt.tz_convert('UTC')
    return out

def convertir_fechas(df):
    out = df.copy()
    fecha_cols = ['fecha_ingreso', 'fecha_salida', 'fecha_vencimiento']
    for col in fecha_cols:
        original = out[col].str.strip()
        iso = pd.to_datetime(original, format='%Y-%m-%d', errors='coerce')
        dmy = pd.to_datetime(original, format='%d/%m/%Y', errors='coerce')
        out[col] = iso.fillna(dmy).dt.strftime('%Y-%m-%d').replace('', pd.NA)
        out[col + '_conversion_determinista'] = iso.isna() & dmy.notna()
        invalid = original.ne('') & out[col].isna()
        out[col + '_conversion_invalida'] = invalid
        out = add_error(out, invalid, col + ':fecha_invalida')
        out['fue_transformada'] |= out[col + '_conversion_determinista']
    out['fecha_salida_vacia'] = out['fecha_salida_original'].eq('')
    return out

def validar_dominio(df):
    out = df.copy()
    centro_invalido = ~out['centro_distribucion'].isin(CONFIG['centros_validos'])
    salida_incoherente = out['cantidad_salida'].eq(0) & out['fecha_salida_vacia']
    out['centro_distribucion_valido'] = ~centro_invalido
    out['salida_incoherente'] = salida_incoherente
    out = add_error(out, centro_invalido, 'centro_distribucion:referencia_invalida')
    out = add_error(out, salida_incoherente, 'salida:salida_requerida_con_cantidad_cero')
    out['cantidad_restante_calculada'] = out['cantidad_ingreso'] - out['cantidad_salida'] - out['cantidad_merma']
    out = add_error(out, out['cantidad_restante_calculada'].lt(0), 'cantidades:restante_negativo')
    out['dias_vencimiento_calculado'] = (pd.Timestamp(CONFIG['fecha_corte']) - pd.to_datetime(out['fecha_vencimiento'], errors='coerce')).dt.days
    out['vence_dentro_30_dias'] = out['dias_vencimiento_calculado'].between(0, 30)
    return out

def controlar_duplicados(df):
    out = df.copy()
    clave = ['movimiento_id', 'lote_id', 'producto_id', 'centro_distribucion']
    out['duplicado_clave'] = out.duplicated(clave, keep=False)
    exact = out.duplicated(clave + CONFIG['columnas_obligatorias'][4:], keep=False)
    out['duplicado_exacto'] = exact
    out['conflicto_clave'] = out['duplicado_clave'] & ~exact
    out['conservar_copia_duplicada'] = exact & ~out.duplicated(clave + CONFIG['columnas_obligatorias'][4:], keep='first')
    out = add_error(out, out['conflicto_clave'], 'duplicado_clave:duplicado_conflictivo')
    out = add_error(out, out['conservar_copia_duplicada'], 'duplicado_exacto:copia_resuelta_con_trazabilidad')
    out['motivos_transformacion'] = np.where(out['conservar_copia_duplicada'], 'duplicado_exacto:copia_conservada_con_trazabilidad', out['motivos_transformacion'])
    return out

def enriquecimiento_auxiliar(df):
    out = df.copy()
    productos_ids = set(productos_aux['producto_id'].str.strip().str.upper())
    out['producto_id_corresponde_productos_silver'] = out['producto_id'].isin(productos_ids)
    out['fecha_vencimiento_fuente'] = np.where(out['fecha_vencimiento'].notna(), 'bronze_observada', 'no_disponible_wms')
    out['wms_lote_id_disponible'] = 'lote_id' in wms_aux.columns
    return out

def calcular_calidad(df):
    out = df.copy()
    out['calidad_motivo'] = out['errores_bloqueantes'].replace('', 'sin_incidencias')
    out['calidad_estado'] = np.where(out['errores_bloqueantes'].eq(''), 'valida', 'cuarentena')
    out['conteo_transformaciones'] = out['fue_transformada'].astype(int)
    out['conteo_imputaciones'] = out['fue_imputada'].astype(int)
    out['imputacion_metodo'] = ''
    out['imputacion_motivo'] = ''
    return out

work = bronze.pipe(preparar).pipe(normalizar_texto).pipe(convertir_numericas).pipe(manejar_timestamps_utc).pipe(convertir_fechas).pipe(validar_dominio).pipe(controlar_duplicados).pipe(enriquecimiento_auxiliar).pipe(calcular_calidad)
silver = work.loc[work['errores_bloqueantes'].eq('') & ~work['conservar_copia_duplicada']].copy()
quarantine = work.loc[work['errores_bloqueantes'].ne('') | work['conservar_copia_duplicada']].copy()
print('Silver', len(silver), 'Cuarentena', len(quarantine))
print('Motivos cuarentena:', quarantine['errores_bloqueantes'].fillna('').replace('', 'duplicado_exacto:copia_conservada_con_trazabilidad').value_counts().to_dict())
print('Silver estados:', silver['calidad_estado'].value_counts().to_dict())
print('Cuarentena estados:', quarantine['calidad_estado'].value_counts().to_dict())

Silver 5980 Cuarentena 60
Motivos cuarentena: {'duplicado_exacto:copia_resuelta_con_trazabilidad': 40, 'cantidad_ingreso:conversion_invalida_o_centinela;cantidad_ingreso:valor_no_numerico_o_negativo': 10, 'cantidad_merma:valor_no_numerico_o_negativo': 5, 'fecha_salida:fecha_invalida': 5}
Silver estados: {'valida': 5980}
Cuarentena estados: {'cuarentena': 60}


In [4]:
silver.to_csv(PATHS['silver'], index=False, encoding='utf-8')
quarantine.to_csv(PATHS['quarantine'], index=False, encoding='utf-8')
assert len(bronze) == len(silver) + len(quarantine)
assert set(silver['_fila_bronze']).isdisjoint(set(quarantine['_fila_bronze']))
assert set(silver['_fila_bronze']) | set(quarantine['_fila_bronze']) == set(bronze['_fila_bronze'])
assert silver['errores_bloqueantes'].eq('').all()
assert silver[['movimiento_id', 'lote_id', 'producto_id', 'centro_distribucion']].duplicated().sum() == 0
assert silver['fecha_salida_vacia'].eq(False).all()
assert not silver['fecha_ingreso'].isna().any()
print('Controles de conciliación, unicidad y bloqueantes: OK')
print(pd.DataFrame({'bronze': [len(bronze)], 'silver': [len(silver)], 'quarantine': [len(quarantine)]}))

Controles de conciliación, unicidad y bloqueantes: OK
   bronze  silver  quarantine
0    6040    5980          60


In [5]:
report = [
    '# Informe B2S 05 - Andinalog Inventory Tracking', '',
    '## Objetivo, entidad y granularidad',
    'Conversión auditada de movimientos de inventario desde Bronze hacia Silver y cuarentena.',
    f"- Entidad: {CONFIG['entidad']}.",
    f"- Granularidad: {CONFIG['granularidad']}.", '',
    '## Perfil Bronze de esta ejecución',
    f"- Filas Bronze: {perfil['filas']}.",
    f"- Columnas: {perfil['columnas']}.",
    f"- Valores vacíos: {perfil['nulos_o_vacios']}.",
    f"- Filas con movimiento_id duplicado: {perfil['duplicados_movimiento_id']}.",
    f"- Filas con lote_id duplicado: {perfil['duplicados_lote_id']}.",
    f"- Centros observados: {perfil['centros_observados']}.",
    f"- Productos observados: {perfil['productos_observados']}.", '',
    '## Reglas y transformaciones',
    '- Las fechas calendario se conservan sin desplazamiento de zona horaria.',
    '- Los timestamps con hora se interpretan en America/La_Paz y se convierten a UTC; en esta fuente no se observaron timestamps con hora.',
    '- Se acepta ISO y se corrige de forma determinista DD/MM/YYYY cuando la fecha es válida.',
    '- Los centros se validan directamente contra el catálogo configurado.',
    '- Los valores originales se conservan antes de convertir texto, fechas y números.',
    '- Los duplicados exactos se resuelven conservando una fila y enviando la copia con trazabilidad.',
    '- Los duplicados conflictivos se envían íntegramente a cuarentena.', '',
    '## Integridad referencial y enriquecimiento',
    '- Silver WMS no contiene lote_id; por lo tanto no se puede recuperar fecha_vencimiento desde esa fuente.',
    '- Los vencimientos observados se conservan como bronze_observada; los vacíos se registran como no_disponible_wms.',
    '- La correspondencia con productos Silver es informativa y no bloquea la fila.',
    '- La bandera vence_dentro_30_dias usa la fecha de corte fija declarada y no es error de calidad.', '',
    '## Resultado y conciliación',
    f"- Silver: {len(silver)} filas.",
    f"- Cuarentena: {len(quarantine)} filas.",
    f"- Conciliación: Bronze {len(bronze)} = Silver {len(silver)} + cuarentena {len(quarantine)}.",
    f"- Fecha de corte: {CONFIG['fecha_corte']}.", '',
    '## Contradicciones documentadas del plan',
    '- El plan aprobado estimaba 537 registros; la inspección real encontró 6040 registros.',
    '- El plan Approved preveía salidas vacías ycantidad_salida mayor que cantidad_ingreso; no se observaron en la fuente actual.',
    '- WMS Silver no expone lote_id, por lo que no se realiza la recuperación de vencimiento prevista.', '',
    '## Archivos generados',
    f"- `{CONFIG['rutas']['notebook']}`",
    f"- `{CONFIG['rutas']['silver']}`",
    f"- `{CONFIG['rutas']['quarantine']}`",
    f"- `{CONFIG['rutas']['informe']}`", '',
    '## Reproducibilidad',
    f"- Fecha UTC: {EXECUTED_AT_UTC}.",
    f"- Python: {platform.python_version()}.",
    f"- pandas: {pd.__version__}.",
    '- Bronze se lee como texto y no se modifica.',
    '- Ejecutar las celdas en orden; los controles se realizan sobre los CSV persistidos.'
]
PATHS['informe'].write_text('\n'.join(report) + '\n', encoding='utf-8')
print('Informe generado:', PATHS['informe'])

Informe generado: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\informes\bronze_silver\Informe_B2S_05_Andinalog_Inventory_Tracking.md
